# Synthetic Temporal Graph Notebook

This notebook is set up so that **only one cell needs to change** when you want to try a different synthetic dataset.

## Workflow
1. Edit the **generator cell** to create a new synthetic dataset.
2. Run the rest of the notebook unchanged.
3. The downstream cells will:
   - validate the generated data
   - visualize it
   - save it to JSON
   - load it through a PyTorch-compatible dataset class
   - show example timestep objects in the format your model expects

## Required output format from the generator cell
The generator cell must define a variable named `synthetic_data` with this structure:

```python
{
    "x_global": [[...], [...], ...],      # shape [num_nodes, embed_dim]
    "id_to_entity": {"0": "A0", ...},     # optional but recommended
    "id_to_rel": {"0": "rel1", "1": "rel2"},
    "timesteps": [
        {"edges": [{"src": 0, "dst": 10, "rel": 0}, ...]},
        {"edges": [...]},
        ...
    ]
}
```

Everything after the generator cell assumes only that format.
Each edge contains a rel_time field which is how long it has existed



In [1]:
# Imports and global config

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Union

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import Tensor
from torch.utils.data import Dataset

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)

OUT_DIR = Path("synthetic_data")
OUT_DIR.mkdir(exist_ok=True)

json_name = "2ent_2rel.json"

JSON_PATH = OUT_DIR / json_name

print(f"Saving outputs under: {OUT_DIR.resolve()}")

Saving outputs under: C:\Users\jaden\OneDrive\Yale Classes\CPSC 4520\Temporal_Knowledge_Graph_GNN\data_exploration\synthetic_data


## Replace only this next cell when you want a different synthetic process

In [3]:
# GENERATE ALL SYNTHETIC DATASETS
# sizes:
#   small  = 40 total nodes
#   medium = 100 total nodes
#   large  = 200 total nodes

import json
import numpy as np
from pathlib import Path

SEED = 7
rng = np.random.default_rng(SEED)

OUT_DIR = Path("synthetic_data")
OUT_DIR.mkdir(exist_ok=True)

EMBED_DIM = 64
T = 300



def make_nodes(num_types, total_nodes):
    nodes_per_type = total_nodes // num_types

    nodes = []
    type_to_nodes = {}

    for t in range(num_types):
        prefix = chr(ord("A") + t)
        type_to_nodes[prefix] = []

        for i in range(nodes_per_type):
            n = f"{prefix}{i}"
            nodes.append(n)
            type_to_nodes[prefix].append(n)

    entity_to_id = {n: i for i, n in enumerate(nodes)}
    id_to_entity = {i: n for n, i in entity_to_id.items()}
    node_types = {entity_to_id[n]: n[0] for n in nodes}

    return nodes, type_to_nodes, entity_to_id, id_to_entity, node_types


def relation_names(n):
    base = [
        "contact", "negotiate", "bonded", "strained", "recovery",
        "conflict", "repair", "maintain", "decay", "scale",
    ]
    return {i: base[i] for i in range(n)}


def build_paths(num_rel):
    if num_rel == 2:
        return {
            0: [(0, 2), (1, 5)],
        }

    if num_rel == 5:
        return {
            0: [(0, 2), (1, 2), (2, 5), (3, 2), (4, 2)],
            1: [(0, 2), (1, 3), (2, 4), (4, 3)],
            2: [(0, 1), (3, 3), (1, 2), (2, 5)],
        }

    if num_rel == 9:
        return {
            0: [(0, 2), (1, 3), (3, 6), (7, 4)],
            1: [(0, 2), (1, 2), (5, 3), (6, 2), (3, 4)],
            2: [(0, 1), (2, 3), (4, 5), (8, 3)],
            3: [(0, 2), (5, 2), (6, 2), (7, 5), (8, 2)],
        }

    return {
        0: [(0, 2), (1, 3), (2, 4), (3, 5), (8, 3)],
        1: [(0, 1), (4, 3), (5, 4), (6, 3), (9, 2)],
        2: [(1, 2), (2, 2), (7, 6), (8, 3)],
        3: [(0, 2), (3, 3), (5, 3), (6, 2), (7, 4), (9, 2)],
        4: [(4, 2), (1, 3), (2, 5), (8, 2)],
    }


def allowed_edge(s_type, d_type, num_types):
    if num_types == 2:
        return s_type != d_type

    return (ord(s_type) - ord("A") + 1) % num_types == (ord(d_type) - ord("A"))


def generate(name, num_types, num_rel, total_nodes, size_name, noise=False):
    nodes, type_to_nodes, entity_to_id, id_to_entity, node_types = make_nodes(
        num_types=num_types,
        total_nodes=total_nodes,
    )

    id_to_rel = relation_names(num_rel)
    paths = build_paths(num_rel)

    x_global = rng.normal(size=(len(nodes), EMBED_DIM)).astype(float)

    for node in nodes:
        nid = entity_to_id[node]
        typ = ord(node[0]) - ord("A")
        idx = int(node[1:])

        x_global[nid, 0] = typ / max(1, num_types - 1)
        x_global[nid, 1] = (idx % 5) / 5
        x_global[nid, 2] = (idx % 11) / 11

    pairs = []
    for s in nodes:
        for d in nodes:
            if s != d and allowed_edge(s[0], d[0], num_types):
                pairs.append((s, d))

    pair_path = {p: -1 for p in pairs}
    pair_idx = {p: 0 for p in pairs}
    pair_time = {p: 0 for p in pairs}

    def choose_path(s, d):
        return (entity_to_id[s] * 31 + entity_to_id[d] * 17) % len(paths)

    def start_prob():
        if size_name == "small":
            return 0.05 if num_types == 2 else 0.03 if num_types == 5 else 0.02
        if size_name == "medium":
            return 0.025 if num_types == 2 else 0.015 if num_types == 5 else 0.01
        return 0.0125 if num_types == 2 else 0.008 if num_types == 5 else 0.005

    timesteps = []

    for t in range(T):
        edges = []
        labels = []

        for s, d in pairs:
            pid = pair_path[(s, d)]
            sid = entity_to_id[s]
            did = entity_to_id[d]

            if pid != -1:
                idx = pair_idx[(s, d)]
                t_phase = pair_time[(s, d)]

                rel, dur = paths[pid][idx]
                rt = t_phase + 1

                labels.append({
                    "src": int(sid),
                    "dst": int(did),
                    "rel": int(rel),
                    "rel_time": float(rt),
                })

                if noise and rng.random() < 0.02:
                    continue

                emit_rel = int(rel)
                emit_time = float(rt)

                if noise and rng.random() < 0.02:
                    emit_rel = int(rng.integers(0, num_rel))

                if noise and rng.random() < 0.03:
                    emit_time = float(max(1, rt + int(rng.choice([-1, 1]))))

                edges.append({
                    "src": int(sid),
                    "dst": int(did),
                    "rel": int(emit_rel),
                    "rel_time": float(emit_time),
                })

            else:
                if noise and rng.random() < 0.001:
                    edges.append({
                        "src": int(sid),
                        "dst": int(did),
                        "rel": int(rng.integers(0, num_rel)),
                        "rel_time": float(rng.integers(1, 5)),
                    })

        timesteps.append({
            "edges": edges,
            "labels": labels,
        })

        new_path = pair_path.copy()
        new_idx = pair_idx.copy()
        new_time = pair_time.copy()

        for s, d in pairs:
            pid = pair_path[(s, d)]

            if pid == -1:
                if rng.random() < start_prob():
                    new_path[(s, d)] = choose_path(s, d)
                    new_idx[(s, d)] = 0
                    new_time[(s, d)] = 0

            else:
                if noise:
                    r = rng.random()

                    if r < 0.01:
                        new_path[(s, d)] = -1
                        new_idx[(s, d)] = 0
                        new_time[(s, d)] = 0
                        continue

                    if r < 0.04:
                        continue

                    if r < 0.07:
                        new_idx[(s, d)] = min(new_idx[(s, d)] + 1, len(paths[pid]) - 1)
                        new_time[(s, d)] = 0
                        continue

                idx = pair_idx[(s, d)]
                t_phase = pair_time[(s, d)]
                rel, dur = paths[pid][idx]

                if t_phase + 1 < dur:
                    new_time[(s, d)] = t_phase + 1
                else:
                    nxt = idx + 1

                    if nxt >= len(paths[pid]):
                        new_path[(s, d)] = -1
                        new_idx[(s, d)] = 0
                        new_time[(s, d)] = 0
                    else:
                        new_idx[(s, d)] = nxt
                        new_time[(s, d)] = 0

        pair_path = new_path
        pair_idx = new_idx
        pair_time = new_time

    data = {
        "x_global": x_global.tolist(),
        "id_to_entity": {str(k): v for k, v in id_to_entity.items()},
        "id_to_rel": {str(k): v for k, v in id_to_rel.items()},
        "node_types": {str(k): v for k, v in node_types.items()},
        "timesteps": timesteps,
        "meta": {
            "name": name,
            "size": size_name,
            "total_nodes": int(total_nodes),
            "nodes_per_type": int(total_nodes // num_types),
            "num_types": int(num_types),
            "num_relations": int(num_rel),
            "embed_dim": int(EMBED_DIM),
            "T": int(T),
            "noise": bool(noise),
            "paths": {
                str(pid): [
                    {
                        "rel": int(rel),
                        "rel_name": id_to_rel[int(rel)],
                        "duration": int(duration),
                    }
                    for rel, duration in path
                ]
                for pid, path in paths.items()
            },
            "edge_restriction": (
                "full directed bipartite across A and B"
                if num_types == 2
                else "directed cycle: each type connects only to the next type"
            ),
        },
    }

    noise_suffix = "_noise" if noise else ""
    path = OUT_DIR / f"{name}{noise_suffix}_{size_name}.json"

    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, allow_nan=False)

    print(f"saved: {path}")


configs = [
    ("2ent_2rel", 2, 2),
    ("2ent_5rel", 2, 5),
    ("2ent_9rel", 2, 9),
    ("5ent_5rel", 5, 5),
    ("10ent_10rel", 10, 10),
]

SIZE_TO_TOTAL_NODES = {
    "small": 40,
}


for size_name, total_nodes in SIZE_TO_TOTAL_NODES.items():
    for name, num_types, num_rel in configs:
        generate(name, num_types, num_rel, total_nodes, size_name, noise=False)
        generate(name, num_types, num_rel, total_nodes, size_name, noise=True)

print("DONE")

saved: synthetic_data\2ent_2rel_small.json
saved: synthetic_data\2ent_2rel_noise_small.json
saved: synthetic_data\2ent_5rel_small.json
saved: synthetic_data\2ent_5rel_noise_small.json
saved: synthetic_data\2ent_9rel_small.json
saved: synthetic_data\2ent_9rel_noise_small.json
saved: synthetic_data\5ent_5rel_small.json
saved: synthetic_data\5ent_5rel_noise_small.json
saved: synthetic_data\10ent_10rel_small.json
saved: synthetic_data\10ent_10rel_noise_small.json
DONE


In [3]:
# Validation and quick summary

required_keys = {"x_global", "id_to_rel", "timesteps"}
missing = required_keys - set(synthetic_data.keys())
if missing:
    raise ValueError(f"synthetic_data missing required keys: {missing}")

xg = np.array(synthetic_data["x_global"], dtype=np.float32)
if xg.ndim != 2:
    raise ValueError(f"x_global must be 2D, got shape {xg.shape}")

num_nodes, embed_dim = xg.shape
num_timesteps = len(synthetic_data["timesteps"])
num_relations = len(synthetic_data["id_to_rel"])

print("Validation passed")
print(f"x_global shape: {xg.shape}")
print(f"num_relations:  {num_relations}")
print(f"num_timesteps:  {num_timesteps}")

edge_counts = [len(step.get("edges", [])) for step in synthetic_data["timesteps"]]
print(f"edge count range: {min(edge_counts)} .. {max(edge_counts)}")

NameError: name 'synthetic_data' is not defined

In [ ]:
# Save synthetic data to JSON

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(synthetic_data, f, indent=2)

print(f"Saved JSON to: {JSON_PATH.resolve()}")

Saved JSON to: C:\Users\jaden\OneDrive\Yale Classes\CPSC 4520\Temporal_Knowledge_Graph_GNN\data_exploration\synthetic_data\2ent_2rel.json


In [ ]:
import math
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
import numpy as np


def plot_temporal_graphs_general(
    synthetic_data,
    timesteps=None,
    cols=3,
    figsize_per_plot=(7, 5),
    layout="spring",
    fixed_layout=True,
    active_only=True,
    show_node_labels=True,
    show_edge_labels=False,
    show_rel_time=False,
    node_size=800,
    font_size=8,
    seed=7,
    multipartite_key="node_types",
):
    """
    General visualization for temporal synthetic graph snapshots.

    Parameters
    ----------
    synthetic_data : dict
        Dataset JSON-like object with keys such as:
          - id_to_entity
          - id_to_rel
          - timesteps
          - optionally node_types
    timesteps : list[int] | None
        Which timesteps to plot. If None, plots all.
    cols : int
        Number of subplot columns.
    figsize_per_plot : tuple
        Size per subplot.
    layout : str
        One of: 'spring', 'kamada_kawai', 'shell', 'circular', 'multipartite'
    fixed_layout : bool
        If True, computes positions once on the union graph and reuses them.
        Strongly recommended for comparing timesteps.
    active_only : bool
        If True, only draw nodes active in that timestep.
    show_node_labels : bool
        Whether to draw node names.
    show_edge_labels : bool
        Whether to draw relation labels on edges.
    show_rel_time : bool
        If True and edge has 'rel_time', append it to edge label.
    node_size : int
        Node marker size.
    font_size : int
        Base font size.
    seed : int
        Random seed for layout reproducibility.
    multipartite_key : str
        Key in synthetic_data containing node grouping info. Example: "node_types".
    """

    id_to_entity = {int(k): v for k, v in synthetic_data["id_to_entity"].items()}
    id_to_rel = {int(k): v for k, v in synthetic_data["id_to_rel"].items()}
    steps = synthetic_data["timesteps"]

    if timesteps is None:
        timesteps = list(range(len(steps)))

    # Optional grouping info for multipartite layouts
    raw_groups = synthetic_data.get(multipartite_key, None)
    node_groups = None
    if raw_groups is not None:
        node_groups = {int(k): str(v) for k, v in raw_groups.items()}

    # Build union graph once for consistent layout and relation discovery
    union_graph = nx.DiGraph()
    union_graph.add_nodes_from(id_to_entity.keys())

    relation_ids_present = set()
    for step in steps:
        for e in step.get("edges", []):
            src = int(e["src"])
            dst = int(e["dst"])
            rel = int(e["rel"])
            union_graph.add_edge(src, dst)
            relation_ids_present.add(rel)

    relation_ids_present = sorted(relation_ids_present)

    # Node colors
    if node_groups is not None:
        unique_groups = sorted(set(node_groups.values()))
        group_cmap = cm.get_cmap("tab10", max(len(unique_groups), 1))
        group_to_color = {
            g: group_cmap(i) for i, g in enumerate(unique_groups)
        }
        node_color_map = {
            nid: group_to_color.get(node_groups.get(nid, "default"), (0.8, 0.8, 0.8, 1.0))
            for nid in id_to_entity
        }
    else:
        node_color_map = {nid: "#bdbdbd" for nid in id_to_entity}

    # Edge colors by relation
    rel_cmap = cm.get_cmap("tab20", max(len(relation_ids_present), 1))
    rel_to_color = {
        rel: rel_cmap(i) for i, rel in enumerate(relation_ids_present)
    }

    def compute_positions(G):
        if layout == "spring":
            return nx.spring_layout(G, seed=seed)
        elif layout == "kamada_kawai":
            return nx.kamada_kawai_layout(G)
        elif layout == "circular":
            return nx.circular_layout(G)
        elif layout == "shell":
            if node_groups is not None:
                unique_groups = sorted(set(node_groups.values()))
                shells = []
                for g in unique_groups:
                    shell = [nid for nid in sorted(G.nodes()) if node_groups.get(nid) == g]
                    if shell:
                        shells.append(shell)
                if shells:
                    return nx.shell_layout(G, shells)
            return nx.shell_layout(G)
        elif layout == "multipartite":
            if node_groups is None:
                raise ValueError(
                    "layout='multipartite' requires synthetic_data['node_types'] "
                    "or another grouping field."
                )
            H = G.copy()
            for nid in H.nodes():
                H.nodes[nid]["subset"] = node_groups.get(nid, "unknown")
            return nx.multipartite_layout(H, subset_key="subset")
        else:
            raise ValueError(
                f"Unknown layout '{layout}'. "
                f"Choose from spring, kamada_kawai, circular, shell, multipartite."
            )

    # Reuse one layout across snapshots if requested
    if fixed_layout:
        base_pos = compute_positions(union_graph)
    else:
        base_pos = None

    n = len(timesteps)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(cols * figsize_per_plot[0], rows * figsize_per_plot[1]),
        squeeze=False,
    )
    axes = axes.flatten()

    for plot_idx, t in enumerate(timesteps):
        ax = axes[plot_idx]
        step = steps[t]

        G = nx.DiGraph()
        G.add_nodes_from(id_to_entity.keys())

        for e in step.get("edges", []):
            src = int(e["src"])
            dst = int(e["dst"])
            rel = int(e["rel"])
            rel_time = float(e.get("rel_time", 0.0))
            G.add_edge(src, dst, rel=rel, rel_time=rel_time)

        if active_only:
            active_nodes = sorted(set(u for u, _ in G.edges()) | set(v for _, v in G.edges()))
            H = G.subgraph(active_nodes).copy()
        else:
            H = G.copy()

        if len(H.nodes()) == 0:
            ax.set_title(f"Timestep {t} | no active edges")
            ax.axis("off")
            continue

        if fixed_layout:
            pos = {nid: base_pos[nid] for nid in H.nodes()}
        else:
            pos = compute_positions(H)

        # Draw nodes
        nx.draw_networkx_nodes(
            H,
            pos,
            node_color=[node_color_map.get(nid, "#bdbdbd") for nid in H.nodes()],
            node_size=node_size,
            edgecolors="black",
            linewidths=0.8,
            ax=ax,
        )

        if show_node_labels:
            nx.draw_networkx_labels(
                H,
                pos,
                labels={nid: id_to_entity[nid] for nid in H.nodes()},
                font_size=font_size,
                ax=ax,
            )

        # Draw relation groups separately
        for rel in relation_ids_present:
            rel_edges = [(u, v) for u, v, d in H.edges(data=True) if d["rel"] == rel]
            if not rel_edges:
                continue

            nx.draw_networkx_edges(
                H,
                pos,
                edgelist=rel_edges,
                edge_color=[rel_to_color[rel]],
                width=2.0,
                alpha=0.85,
                arrows=True,
                arrowsize=16,
                ax=ax,
            )

        if show_edge_labels:
            edge_labels = {}
            for u, v, d in H.edges(data=True):
                rel_name = id_to_rel.get(d["rel"], str(d["rel"]))
                if show_rel_time:
                    edge_labels[(u, v)] = f"{rel_name}\nrt={d['rel_time']:.1f}"
                else:
                    edge_labels[(u, v)] = rel_name

            nx.draw_networkx_edge_labels(
                H,
                pos,
                edge_labels=edge_labels,
                font_size=max(font_size - 1, 6),
                rotate=False,
                bbox={"alpha": 0.7, "color": "white", "pad": 0.15},
                ax=ax,
            )

        ax.set_title(f"Timestep {t} | edges={H.number_of_edges()}", fontsize=11)
        ax.axis("off")

    for j in range(n, len(axes)):
        axes[j].axis("off")

    # Add relation legend once
    handles = []
    labels = []
    for rel in relation_ids_present:
        handles.append(
            plt.Line2D([0], [0], color=rel_to_color[rel], lw=2)
        )
        labels.append(id_to_rel.get(rel, str(rel)))

    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(len(handles), 6), frameon=False)

    plt.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()

In [ ]:
plot_temporal_graphs_general(synthetic_data, timesteps=[0, 1, 2, 3, 4, 5])

C:\Users\jaden\AppData\Local\Temp\ipykernel_55336\3379695750.py:95: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  group_cmap = cm.get_cmap("tab10", max(len(unique_groups), 1))
C:\Users\jaden\AppData\Local\Temp\ipykernel_55336\3379695750.py:107: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rel_cmap = cm.get_cmap("tab20", max(len(relation_ids_present), 1))
